In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

In [12]:
import torch
import requests
import tempfile
from pathlib import Path
import shutil
import subprocess
from time import perf_counter_ns


# Video source: https://www.pexels.com/video/dog-eating-854132/
# License: CC0. Author: Coverr.
url = "https://videos.pexels.com/video-files/854132/854132-sd_640_360_25fps.mp4"
response=requests.get(url, headers={'User-Agent':''})
if response.status_code!=200: raise RuntimeError(f'Failed to download video. {response.status_code=}')

temp_dir=Path('D:/results/temp')
temp_dir.mkdir(parents=True, exist_ok=True)
short_video_path=temp_dir/"short_video.mp4"
with open(short_video_path, 'wb') as f: 
    for chunk in response.iter_content(): f.write(chunk) # loop through 1 byte a time
long_video_path=temp_dir/"long_video.mp4"
ffmpeg_command=[
    "ffmpeg",
    "-stream_loop", "99", # repeat video 100 times
    "-i", f"{short_video_path}",
    "-c", "copy",
    f"{long_video_path}"
]
subprocess.run(ffmpeg_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

from torchcodec.decoders import VideoDecoder
print(f'Short video duration: {VideoDecoder(short_video_path).metadata.duration_seconds} seconds')
print(f'Long video duration: {VideoDecoder(long_video_path).metadata.duration_seconds/60} minutes')


Short video duration: 13.8 seconds
Long video duration: 23.0 minutes


In [14]:
def bench(f, average_over=50, warmup=2, **f_kwargs):
    for _ in range(warmup): f(**f_kwargs)

    times=[]
    for _ in range(average_over):
        start=perf_counter_ns()
        f(**f_kwargs)
        end=perf_counter_ns()
        times.append(end-start)

    times=torch.tensor(times)*1e-6 # nanoseconds to miliseconds
    std=times.std().item()
    med=times.median().item()
    print(f'{med=:.2f} +- {std:.2f} ms')

print("Creating a VideoDecoder object with seek_mokde='exact' on a short video: ")
bench(VideoDecoder, source=short_video_path, seek_mode='exact')
print("Creating a VideoDecoder object with seek_mokde='approximate' on a short video: ")
bench(VideoDecoder, source=short_video_path, seek_mode='approximate')
print()
print("Creating a VideoDecoder object with seek_mokde='exact' on a long video: ")
bench(VideoDecoder, source=long_video_path, seek_mode='exact')
print("Creating a VideoDecoder object with seek_mokde='approximate' on a long video: ")
bench(VideoDecoder, source=long_video_path, seek_mode='approximate')

Creating a VideoDecoder object with seek_mokde='exact' on a short video: 
med=5.21 +- 0.09 ms
Creating a VideoDecoder object with seek_mokde='approximate' on a short video: 
med=4.63 +- 0.18 ms

Creating a VideoDecoder object with seek_mokde='exact' on a long video: 
med=77.51 +- 1.41 ms
Creating a VideoDecoder object with seek_mokde='approximate' on a long video: 
med=6.84 +- 0.15 ms


In [16]:
from torchcodec import samplers

def sample_clips(seek_mode):
    return samplers.clips_at_random_indices(decoder=VideoDecoder(source=long_video_path, seek_mode=seek_mode),
                                            num_clips=5, num_frames_per_clip=2)
print('Sampling clips with seek_mode=exact: ')
bench(sample_clips, seek_mode='exact')
print('Sampling clips with seek_mode=approximate: ')
bench(sample_clips, seek_mode='approximate')

Sampling clips with seek_mode=exact: 
med=180.70 +- 18.54 ms
Sampling clips with seek_mode=approximate: 
med=113.71 +- 18.81 ms


In [17]:
print('Metadata of short video with seek_mode=exact: ')
print(VideoDecoder(short_video_path, seek_mode='exact').metadata)
print('Metadata of short video with seek_mode=approximate: ')
print(VideoDecoder(short_video_path, seek_mode='approximate').metadata)

exact_decoder=VideoDecoder(short_video_path, seek_mode='exact')
approx_decoder=VideoDecoder(short_video_path, seek_mode='approximate')
for i in range(len(exact_decoder)):
    torch.testing.assert_close(exact_decoder.get_frame_at(i).data, approx_decoder.get_frame_at(i).data, atol=0., rtol=0.)
print('Frame seeking is the same for this video')

Metadata of short video with seek_mode=exact: 
VideoStreamMetadata:
  duration_seconds_from_header: 13.8
  begin_stream_seconds_from_header: 0.0
  bit_rate: 505790.0
  codec: h264
  stream_index: 0
  duration_seconds: 13.8
  begin_stream_seconds: 0.0
  begin_stream_seconds_from_content: 0.0
  end_stream_seconds_from_content: 13.8
  width: 640
  height: 360
  num_frames_from_header: 345
  num_frames_from_content: 345
  average_fps_from_header: 25.0
  pixel_aspect_ratio: 1
  end_stream_seconds: 13.8
  num_frames: 345
  average_fps: 25.0

Metadata of short video with seek_mode=approximate: 
VideoStreamMetadata:
  duration_seconds_from_header: 13.8
  begin_stream_seconds_from_header: 0.0
  bit_rate: 505790.0
  codec: h264
  stream_index: 0
  duration_seconds: 13.8
  begin_stream_seconds: 0.0
  begin_stream_seconds_from_content: None
  end_stream_seconds_from_content: None
  width: 640
  height: 360
  num_frames_from_header: 345
  num_frames_from_content: None
  average_fps_from_header: 25.